EDA functions (production ready)

In [10]:
import pandas as pd
import numpy as np

def comprehensive_eda(df, target_col=None):
  """
  Complete EDA pipeline following best practices.
  Returns insights dictionary for documentation.
  """
  insights = {}

  # Shape and types
  print("=" * 60)
  print("DATASET OVERVIEW")
  print("=" * 60)
  print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} cols" )
  print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

  numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
  categorical_cols = df.select_dtypes(include=['object','category']).columns.tolist()
  print(f"Numerical cols ({len(numeric_cols)}): {numeric_cols[:5]} {'...' if len(numeric_cols) > 5 else ''}")
  print(f"Categorical cols ({len(categorical_cols)}): {categorical_cols[:5]} {'...' if len(categorical_cols) > 5 else ''}")

  insights['shape'] = df.shape
  insights['numeric_cols'] = numeric_cols
  insights['categorical_cols'] = categorical_cols

  # Missing Values
  print("\n" + "=" * 60)
  print("MISSING VALUES")
  print("=" * 60)
  missing = df.isnull().sum()
  missing_pct = (missing / len(df) * 100).round(2)
  missing_df = (pd.DataFrame({'count':missing, 'percent':missing_pct}))
  missing_df = missing_df[missing_df['count'] > 0].sort_values('count', ascending=False)

  if(len(missing_df) > 0):
    print(missing_df)
    insights['missing'] = missing_df.to_dict()
  else:
    print("No missing values!")
    insights['missing'] = None

  # Numeric Summary + outliers
  print("\n" + "=" * 60)
  print("Numeric Summary with outliers")
  print("=" * 60)

  if(len(numeric_cols) > 0):
    summary = df[numeric_cols].describe().T

    #Add outlier counts using IQR method
    outlier_count = []
    for col in numeric_cols:
      Q1 = df[col].quantile(0.25)
      Q3 = df[col].quantile(0.75)
      IQR = Q3 - Q1
      outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
      outlier_count.append(outliers)
    summary['outliers_IQR'] = outlier_count
    print(summary[['mean','std','min','50%','max','outliers_IQR']])
    insights['numeric_summary'] = summary.to_dict()


    # Target Analysis

    if(target_col and target_col in df.columns):
      print("\n" + "="*60)
      print(f"Target Analysis: {target_col}")
      print("=" * 60)

      if(df[target_col].dtype in ['object', 'category'] or df[target_col].nunique() < 10):
        #Classification target
        value_counts = df[target_col].value_counts()
        print("class distributions")
        print(value_counts)
        print(f"\nClass balance: {value_counts.min()/value_counts.max():.2f} (1.0 == perfectly balanced)")

        insights['target_type'] = 'classification'
        insights['class_balance'] = (value_counts / len(df)).to_dict()
      else:
        # Regression target
        print(df[target_col].describe())
        print(f"\nSkewness: {df[target_col].skew():.2f}")
        insights['target_type'] = 'regression'
        insights['target_skew'] = df[target_col].skew()

  # correlations

  if len(numeric_cols) > 1:
    print("\n" + '=' * 60)
    print("Top Correlations")
    print('=' * 60)

    corr_matrix = df[numeric_cols].corr()

    # Get top correlations (excluding self-correlations)
    corr_pairs = []
    for i, col1 in enumerate(numeric_cols):
      for col2 in numeric_cols[i+1:]:
        corr_pairs.append((col1, col2,corr_matrix.loc[col1,col2]))
    corr_pairs.sort(key = lambda x: abs(x[2]), reverse=True)
    print("Highest Correlations:")
    for col1, col2, corr in corr_pairs[:5]:
      print(f"{col1} <-> {col2}: {corr:.3f}")

    insights['top_correlations'] = corr_pairs[:10]

  print("\n" + "=" * 60)
  print("EDA COMPLETE")
  print("=" * 60)
  return insights

#usage
file_name = "/content/sample_data/messy_ecommerce_customers_10k.csv"
print(f"Loading data from {file_name}")
df = pd.read_csv(file_name)
results = comprehensive_eda(df, target_col='is_premium_member')



Loading data from /content/sample_data/messy_ecommerce_customers_10k.csv
DATASET OVERVIEW
Shape: 10,050 rows x 18 cols
Memory: 7.57 MB
Numerical cols (4): ['age', 'total_orders', 'customer_satisfaction_score', 'cart_abandonment_rate'] 
Categorical cols (13): ['customer_id', 'first_name', 'last_name', 'email', 'gender'] ...

MISSING VALUES
                             count  percent
customer_satisfaction_score    805     8.01
age                            499     4.97
gender                         303     3.01
preferred_category             203     2.02

Numeric Summary with outliers
                                  mean        std   min   50%    max  \
age                          46.010261  17.889612 -48.0  45.0  299.0   
total_orders                 74.272836  43.044398   0.0  74.0  149.0   
customer_satisfaction_score   3.030287   1.464846   1.0   3.0   14.0   
cart_abandonment_rate         0.495129   0.290968   0.0   0.5    1.0   

                             outliers_IQR  
age

Outlier Detection and Handling

IQR and Z-score methods with handling strategies


In [11]:
import pandas as pd
import numpy as np


def detect_outliers(df, columns=None, method='iqr', threshold=1.5):
  """
    Detect outliers using IQR or Z-score method.

    Parameters:
 - method: 'iqr' or 'zscore'
 - threshold: 1.5 for IQR (mild), 3.0 for IQR (extreme), 3 for Z-score

    Returns: DataFrame with outlier flags
    """
  if columns is None:
    columns = df.select_dtypes(include=np.number).columns

  outlier_flags = pd.DataFrame(index=df.index)

  for col in columns:
    if method == 'iqr':
      Q1 = df[col].quantile(0.25)
      Q3 = df[col].quantile(0.75)
      IQR = Q3 - Q1
      lower = Q1 - threshold * IQR
      upper = Q3 + threshold * IQR
      outlier_flags[f'{col}_outlier'] = (df[col] < lower) | (df[col] > upper)
    elif method == 'zscore':
      mean = df[col].mean()
      std = df[col].std()
      z_scores = (df[col] - mean) / std
      outlier_flags[f'{col}_outlier'] = np.abs(z_scores) > threshold
  #Summary

  print("Outlier Detection Summary")
  print("-" * 60)

  for col in outlier_flags:
    n_outliers = outlier_flags[col].sum()
    pct = n_outliers / len(df) * 100
    print(f"{col}: {n_outliers} outliers ({pct:1f}%)")

  return outlier_flags

def handle_outliers(df, column, statergy= 'cap', **kwargs):
  df = df.copy()
  if statergy == 'cap':
    lower_pct = kwargs.get('lower',0.01)
    upper_pct = kwargs.get('upper',0.99)
    lower_val = df[column].quantile(lower_pct)
    upper_val = df[column].quantile(upper_pct)
    df[column] = df[column].clip(lower = lower_val, upper = upper_val)
    print(f"Capped {column} to [{lower_val:.2f}, {upper_val:.2f}]")
  elif statergy == 'remove':
    Q1, Q3 = df[column].quantile(0.25,0.75)
    IQR = Q3 - Q1
    mask = (df[column] >= Q1 - 1.5 * IQR) & (df[column] <= Q3 + 1.5 * IQR)
    removed = len(df) - mask.sum()
    df = df[mask]
    print(f"Removed {removed} outlier rows from {column}")
  elif statergy == 'transform':
    df[f"{column}_log"] = np.log1p(df[column].clip(lower = 0))
    print(f"Created log-transformed column: {column}_log")
  elif statergy == 'indicator':
    Q1, Q3 = df[column].quantile(0.25,0.75)
    IQR = Q3 - Q1
    df[f'{column}_is_outlier'] = ((df[column] < Q1 - 1.5 * IQR) |
                                  (df[column] > Q3 + 1.5 * IQR)).astype(int)
    print(f"Created indicator column: {column}_is_outlier")

  return df
# Usage examples:
file_name = "/content/sample_data/messy_ecommerce_customers_10k.csv"
print(f"Loading data from {file_name}")
df = pd.read_csv(file_name)
outlier_flags = detect_outliers(df, method='iqr')
df = handle_outliers(df, 'total_orders', strategy='cap',lower=0.01, upper=0.99)
df = handle_outliers(df, 'age', strategy='remove')
df = handle_outliers(df, 'customer_satisfaction_score', strategy='transform')
df = handle_outliers(df, 'cart_abandonment_rate', strategy='indicator')



Loading data from /content/sample_data/messy_ecommerce_customers_10k.csv
Outlier Detection Summary
------------------------------------------------------------
age_outlier: 20 outliers (0.199005%)
total_orders_outlier: 0 outliers (0.000000%)
customer_satisfaction_score_outlier: 19 outliers (0.189055%)
cart_abandonment_rate_outlier: 0 outliers (0.000000%)
Capped total_orders to [1.00, 148.00]
Capped age to [18.00, 74.00]
Capped customer_satisfaction_score to [1.00, 5.00]
Capped cart_abandonment_rate to [0.01, 0.99]
